# 🧠 Introduction to Artificial Neural Networks (ANN)
### A Hands-On Tutorial for Students

---

**Learning Objectives:**
By the end of this notebook, you will:
1. Understand what a neuron and a neural network are
2. Know how data flows through a network (forward propagation)
3. Understand how a network *learns* (backpropagation + gradient descent)
4. Build and train an ANN on a real dataset
5. Evaluate and visualize the model's performance

**Dataset:** We'll use the **Breast Cancer Wisconsin dataset** — a classic, clean dataset that fits in memory and trains fast.

---

## 📦 Step 0: Install & Import Libraries

We'll use:
- `scikit-learn` — dataset + preprocessing + metrics
- `tensorflow/keras` — building the neural network
- `matplotlib` & `seaborn` — visualizations
- `pandas` & `numpy` — data manipulation

In [ ]:
# Install if needed (uncomment and run once)
# !pip install tensorflow scikit-learn matplotlib seaborn pandas numpy

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, ConfusionMatrixDisplay)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print(f"TensorFlow version: {tf.__version__}")
print("✅ All libraries loaded successfully!")

ModuleNotFoundError: No module named 'tensorflow'

---
## 🔬 Step 1: Understanding the Problem — What Are We Solving?

**Task:** Given measurements from a digitized image of a breast mass (e.g., radius, texture, smoothness...), predict whether the tumor is **Malignant (cancerous)** or **Benign (non-cancerous)**.

This is a **binary classification** problem — perfect for learning ANNs.

In [ ]:
# Load the dataset
data = load_breast_cancer()

# Create a DataFrame for easy exploration
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = Malignant, 1 = Benign
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print("📊 Dataset Overview")
print(f"   Shape: {df.shape}  ({df.shape[0]} samples, {df.shape[1]-2} features)")
print(f"   Classes: {data.target_names}")
print(f"   Class distribution:")
print(df['diagnosis'].value_counts().to_string())
print()
df.drop(columns=['target','diagnosis']).describe().round(2).head(3)

### 📈 Visualize the Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: Count bar chart ---
counts = df['diagnosis'].value_counts()
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=12, fontweight='bold')

# --- Plot 2: Pie chart ---
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportions', fontsize=14, fontweight='bold')

plt.suptitle('Breast Cancer Dataset — Target Variable', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 🔍 Explore Feature Distributions

Let's look at some key features and how they differ between Malignant and Benign tumors.

In [ ]:
key_features = ['mean radius', 'mean texture', 'mean perimeter',
                'mean area', 'mean smoothness', 'mean concavity']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for label, color in [('Malignant', '#e74c3c'), ('Benign', '#2ecc71')]:
        subset = df[df['diagnosis'] == label][feat]
        axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[i].set_title(feat.title(), fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Class', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("💡 Notice: Malignant tumors tend to have larger radius, perimeter, and area.")

### 🌡️ Correlation Heatmap

In [ ]:
# Use only 'mean' features for clarity
mean_features = [c for c in df.columns if 'mean' in c]
corr = df[mean_features].corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Matrix (Mean Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 High correlations (e.g., radius & perimeter) make sense geometrically.")
print("   ANNs can still use correlated features — they learn to weigh them together.")

---
## 🧠 Step 2: The Big Idea — How Does a Neural Network Work?

### The Biological Inspiration

A neural network is loosely inspired by the human brain. The brain has **~86 billion neurons**, each connected to thousands of others. An artificial neuron is a simplified version:

```
  Input 1 (x₁) ──── w₁ ──┐
  Input 2 (x₂) ──── w₂ ──┤──► [ Σ + bias ] ──► Activation ──► Output
  Input 3 (x₃) ──── w₃ ──┘
```

**Each neuron computes:**  
> `output = activation(w₁·x₁ + w₂·x₂ + w₃·x₃ + bias)`

When many neurons are stacked in **layers**, we get a neural network!

In [ ]:
# ─── Visualize a single neuron ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

# Inputs
input_ys = [1.5, 3.0, 4.5]
input_labels = ['mean radius\n(x₁)', 'mean texture\n(x₂)', 'mean area\n(x₃)']
for y, lbl in zip(input_ys, input_labels):
    circ = plt.Circle((1.5, y), 0.55, color='#3498db', zorder=3, linewidth=2)
    ax.add_patch(circ)
    ax.text(1.5, y, lbl, ha='center', va='center', fontsize=6.5,
            color='white', fontweight='bold')

# Neuron body
neuron = plt.Circle((5.5, 3.0), 0.85, color='#e67e22', zorder=3, linewidth=2)
ax.add_patch(neuron)
ax.text(5.5, 3.1, 'Σ + b', ha='center', va='center', fontsize=11, color='white', fontweight='bold')
ax.text(5.5, 2.6, 'activation', ha='center', va='center', fontsize=7, color='#ffe0b2')

# Output
out_circ = plt.Circle((8.5, 3.0), 0.55, color='#2ecc71', zorder=3, linewidth=2)
ax.add_patch(out_circ)
ax.text(8.5, 3.0, 'Output', ha='center', va='center', fontsize=7.5,
        color='white', fontweight='bold')

# Arrows
weights = ['w₁=0.8', 'w₂=0.3', 'w₃=0.6']
for y, w in zip(input_ys, weights):
    ax.annotate('', xy=(4.65, 3.0), xytext=(2.05, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.8))
    mx, my = (2.05 + 4.65) / 2, (y + 3.0) / 2
    ax.text(mx - 0.1, my + 0.15, w, fontsize=8, color='#c0392b', fontweight='bold')

ax.annotate('', xy=(7.95, 3.0), xytext=(6.35, 3.0),
            arrowprops=dict(arrowstyle='->', color='#555', lw=2))

ax.set_title('A Single Artificial Neuron', fontsize=14, fontweight='bold', pad=10)
ax.text(5.5, 0.5, 'output = activation(w₁·x₁ + w₂·x₂ + w₃·x₃ + bias)',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#ecf0f1', edgecolor='gray'))
plt.tight_layout()
plt.show()

### ⚡ Activation Functions

Without an activation function, a neural network is just a **linear transformation** — no matter how many layers. Activations add **non-linearity**, letting the network learn complex patterns.

In [ ]:
x_vals = np.linspace(-5, 5, 300)

activations = {
    'ReLU\n(used in hidden layers)': (np.maximum(0, x_vals), '#e74c3c'),
    'Sigmoid\n(used in output layer)': (1 / (1 + np.exp(-x_vals)), '#3498db'),
    'Tanh': (np.tanh(x_vals), '#2ecc71'),
    'Leaky ReLU': (np.where(x_vals > 0, x_vals, 0.1 * x_vals), '#9b59b6'),
}

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (name, (y, color)) in zip(axes, activations.items()):
    ax.plot(x_vals, y, color=color, linewidth=2.5)
    ax.axhline(0, color='gray', lw=0.8, linestyle='--')
    ax.axvline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Input (z)')
    ax.set_ylabel('Output')
    ax.set_xlim(-5, 5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Common Activation Functions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 We'll use ReLU in hidden layers and Sigmoid in the output layer (binary classification).")

### 🏗️ A Full Neural Network — Layer by Layer

Our network will look like this:

```
INPUT LAYER       HIDDEN LAYER 1    HIDDEN LAYER 2    OUTPUT LAYER
30 features  →→→  16 neurons   →→→  8 neurons    →→→  1 neuron (probability)
                  (ReLU)            (ReLU)             (Sigmoid → 0 or 1)
```

In [ ]:
# ─── Visualize the network architecture ──────────────────────────────────────
def draw_network(layer_sizes, layer_names, ax):
    """Draw a simple neural network diagram."""
    max_neurons = max(layer_sizes)
    colors = ['#3498db', '#e67e22', '#e67e22', '#2ecc71']
    node_radius = 0.35
    x_positions = np.linspace(1, 9, len(layer_sizes))
    node_positions = []

    for idx, (x, n, color, name) in enumerate(zip(x_positions, layer_sizes, colors, layer_names)):
        # Cap display neurons for clarity
        display_n = min(n, 6)
        y_positions = np.linspace(1, 7, display_n)
        node_positions.append((x, y_positions))

        for y in y_positions:
            circ = plt.Circle((x, y), node_radius, color=color,
                              zorder=4, linewidth=1.5, edgecolor='white')
            ax.add_patch(circ)

        # Dots if truncated
        if n > 6:
            ax.text(x, 0.3, f'({n} total)', ha='center', fontsize=8,
                    color='gray', style='italic')

        ax.text(x, 7.7, name, ha='center', fontsize=9,
                fontweight='bold', color='#2c3e50')

    # Draw connections
    for i in range(len(node_positions) - 1):
        x1, ys1 = node_positions[i]
        x2, ys2 = node_positions[i + 1]
        for y1 in ys1:
            for y2 in ys2:
                ax.plot([x1 + node_radius, x2 - node_radius],
                        [y1, y2], 'gray', alpha=0.15, lw=0.8, zorder=1)

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 10)
ax.set_ylim(-0.5, 8.5)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

draw_network(
    layer_sizes=[30, 16, 8, 1],
    layer_names=['Input Layer\n(30 features)', 'Hidden Layer 1\n(16 neurons, ReLU)',
                 'Hidden Layer 2\n(8 neurons, ReLU)', 'Output Layer\n(1 neuron, Sigmoid)'],
    ax=ax
)

# Legend
patches = [
    mpatches.Patch(color='#3498db', label='Input neurons'),
    mpatches.Patch(color='#e67e22', label='Hidden neurons'),
    mpatches.Patch(color='#2ecc71', label='Output neuron'),
]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_title('Our ANN Architecture', fontsize=15, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

---
## 🛠️ Step 3: Prepare the Data

Before feeding data to a neural network, we must:
1. **Split** into training and test sets
2. **Scale** the features (neural networks are sensitive to feature magnitude)

In [ ]:
# ─── Features & Labels ───────────────────────────────────────────────────────
X = data.data
y = data.target  # 1 = Benign, 0 = Malignant

# ─── Train / Test Split (80% / 20%) ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features         : {X_train.shape[1]}")

In [ ]:
# ─── Why scaling matters — visualize before & after ──────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only!
X_test_scaled  = scaler.transform(X_test)        # apply same transform to test

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data_arr, title in zip(axes,
                                [X_train[:, :10], X_train_scaled[:, :10]],
                                ['Before Scaling (first 10 features)',
                                 'After Scaling (first 10 features)']):
    ax.boxplot(data_arr, notch=False, patch_artist=True,
               boxprops=dict(facecolor='#3498db', alpha=0.5),
               medianprops=dict(color='red', lw=2))
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Index')
    ax.set_ylabel('Value')

plt.suptitle('Effect of StandardScaler on Feature Values', fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 After scaling: all features have mean ≈ 0 and std ≈ 1.")
print("   This prevents large-valued features from dominating the learning process.")

---
## 🔨 Step 4: Build the Neural Network

We use **Keras** (part of TensorFlow) — a high-level API that makes building networks simple and readable.

In [ ]:
def build_model(input_dim):
    """Build and compile our ANN."""
    model = keras.Sequential([
        # Input layer (implicit) + Hidden Layer 1
        layers.Dense(16, activation='relu', input_shape=(input_dim,),
                     name='hidden_1'),
        layers.Dropout(0.2, name='dropout_1'),  # regularization

        # Hidden Layer 2
        layers.Dense(8, activation='relu', name='hidden_2'),
        layers.Dropout(0.2, name='dropout_2'),

        # Output Layer — sigmoid gives probability between 0 and 1
        layers.Dense(1, activation='sigmoid', name='output'),
    ], name='ANN_Classifier')

    model.compile(
        optimizer='adam',          # adaptive learning rate
        loss='binary_crossentropy',# for binary classification
        metrics=['accuracy']
    )
    return model

model = build_model(input_dim=X_train_scaled.shape[1])
model.summary()

### 🧮 Count the Parameters

**Why do these numbers make sense?**

| Layer | Neurons | Weights | Biases | Total |
|-------|---------|---------|--------|-------|
| Input → Hidden 1 | 30 → 16 | 30×16=480 | 16 | **496** |
| Hidden 1 → Hidden 2 | 16 → 8 | 16×8=128 | 8 | **136** |
| Hidden 2 → Output | 8 → 1 | 8×1=8 | 1 | **9** |

**Total trainable parameters: 641** — tiny! A single modern LLM has billions.

---
## 🏋️ Step 5: Train the Network

Training = **repeatedly** showing the network examples, calculating how wrong it is (**loss**), and adjusting weights via **backpropagation** + **gradient descent**.

In [ ]:
# Early stopping prevents overfitting and speeds up training
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

history = model.fit(
    X_train_scaled, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.15,     # 15% of training data used for validation
    callbacks=[early_stop],
    verbose=1
)

print(f"\n✅ Training complete! Ran for {len(history.history['loss'])} epochs.")

### 📉 Visualize Training — Loss and Accuracy Curves

These curves tell the story of **how the network learned** over time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

epochs_ran = range(1, len(history.history['loss']) + 1)

# --- Loss ---
axes[0].plot(epochs_ran, history.history['loss'], color='#e74c3c',
             linewidth=2, label='Training Loss')
axes[0].plot(epochs_ran, history.history['val_loss'], color='#e74c3c',
             linewidth=2, linestyle='--', label='Validation Loss')
best_epoch = np.argmin(history.history['val_loss']) + 1
axes[0].axvline(best_epoch, color='gray', linestyle=':', lw=1.5,
                label=f'Best epoch: {best_epoch}')
axes[0].set_title('Loss over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Accuracy ---
axes[1].plot(epochs_ran, history.history['accuracy'], color='#3498db',
             linewidth=2, label='Training Accuracy')
axes[1].plot(epochs_ran, history.history['val_accuracy'], color='#3498db',
             linewidth=2, linestyle='--', label='Validation Accuracy')
axes[1].axvline(best_epoch, color='gray', linestyle=':', lw=1.5,
                label=f'Best epoch: {best_epoch}')
axes[1].set_title('Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.7, 1.02)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("💡 When training and validation curves are close → healthy learning.")
print("   If training accuracy >> validation accuracy → overfitting!")

---
## 📊 Step 6: Evaluate on the Test Set

The test set was **never seen during training** — it's our honest estimate of real-world performance.

In [ ]:
# Get raw probability predictions
y_prob = model.predict(X_test_scaled, verbose=0).flatten()

# Convert probabilities → binary labels (threshold = 0.5)
y_pred = (y_prob >= 0.5).astype(int)

# Overall metrics
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc * 100:.2f}%")
print()
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

### 🔲 Confusion Matrix

A confusion matrix shows **where** the model is right and wrong — crucial for medical applications.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Confusion Matrix ---
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Malignant', 'Benign'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# Annotate the meaning of each cell
tn, fp, fn, tp = cm.ravel()
labels_meaning = [
    [f'True Neg (TN)\n{tn}', f'False Pos (FP)\n{fp}'],
    [f'False Neg (FN)\n{fn}', f'True Pos (TP)\n{tp}']
]
cell_colors = [['#2ecc71', '#e74c3c'], ['#e74c3c', '#2ecc71']]

# --- Breakdown bar chart ---
bars = ['True Negative\n(Correct)', 'False Positive\n(Wrong)', 'False Negative\n(Wrong)', 'True Positive\n(Correct)']
vals = [tn, fp, fn, tp]
colors_bar = ['#2ecc71', '#e74c3c', '#e74c3c', '#2ecc71']
axes[1].bar(bars, vals, color=colors_bar, edgecolor='black', alpha=0.8)
for i, v in enumerate(vals):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontsize=12, fontweight='bold')
axes[1].set_title('Prediction Breakdown', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')

plt.suptitle(f'Model Performance on Test Set  (Accuracy: {test_acc*100:.1f}%)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("⚠️  In medical diagnosis, False Negatives (missed cancer) are especially costly!")

### 📈 ROC Curve — How Good Is Our Model at Separating Classes?

The **ROC curve** plots the True Positive Rate vs False Positive Rate at different thresholds.  
**AUC (Area Under Curve)** = 1.0 is perfect; AUC = 0.5 is random guessing.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- ROC Curve ---
axes[0].plot(fpr, tpr, color='#3498db', lw=2.5,
             label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC=0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#3498db')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.02])
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# --- Prediction Probability Distribution ---
for label, color, name in [(0, '#e74c3c', 'Malignant'), (1, '#2ecc71', 'Benign')]:
    mask = y_test == label
    axes[1].hist(y_prob[mask], bins=20, alpha=0.6, color=color, label=name,
                 edgecolor='white', density=True)

axes[1].axvline(0.5, color='black', linestyle='--', lw=2, label='Decision threshold (0.5)')
axes[1].set_xlabel('Predicted Probability (Benign)', fontsize=11)
axes[1].set_ylabel('Density')
axes[1].set_title('Predicted Probabilities by Class', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Model Discrimination Ability', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"AUC = {roc_auc:.4f} — {'Excellent!' if roc_auc > 0.95 else 'Good' if roc_auc > 0.9 else 'Acceptable'}")

---
## 🔍 Step 7: Understand What the Network Learned

Let's peek inside the network — what did the neurons in the first layer actually learn?

In [ ]:
# Get weights from the first hidden layer
first_layer_weights = model.get_layer('hidden_1').get_weights()[0]  # shape (30, 16)

plt.figure(figsize=(13, 5))
sns.heatmap(
    first_layer_weights.T,
    xticklabels=[f.replace(' ', '\n') for f in data.feature_names],
    yticklabels=[f'Neuron {i+1}' for i in range(16)],
    cmap='RdBu_r', center=0,
    linewidths=0.3,
    annot=False,
    cbar_kws={'label': 'Weight value'}
)
plt.title('Learned Weights — Hidden Layer 1\n(Each row = one neuron, each column = one input feature)',
          fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

print("💡 Red = strong positive weight (feature pushes neuron to fire)")
print("   Blue = strong negative weight (feature inhibits neuron)")
print("   White ≈ 0 = feature mostly ignored by that neuron")

### 🎯 Feature Importance via Weight Magnitude

In [ ]:
# Simple importance: mean absolute weight across all neurons in layer 1
importance = np.mean(np.abs(first_layer_weights), axis=1)
feat_importance = pd.Series(importance, index=data.feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 7))
colors_imp = ['#e74c3c' if v >= feat_importance.quantile(0.75) else '#3498db'
              for v in feat_importance.values]
bars = plt.barh(feat_importance.index, feat_importance.values,
                color=colors_imp, edgecolor='white', alpha=0.85)
plt.xlabel('Mean |Weight| (proxy for importance)', fontsize=11)
plt.title('Feature Importance (Layer 1 Weights)', fontsize=13, fontweight='bold')

red_patch = mpatches.Patch(color='#e74c3c', label='Top 25% most influential')
blue_patch = mpatches.Patch(color='#3498db', label='Others')
plt.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.show()

---
## 🔬 Step 8: Compare Against a Baseline

How much better is our ANN versus a simple rule like **"always predict Benign"**?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Baseline 1: Majority class
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_scaled, y_train)
dummy_acc = dummy.score(X_test_scaled, y_test)

# Baseline 2: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
lr_acc = lr.score(X_test_scaled, y_test)

# Results
results = {
    'Always Benign\n(Dummy)': dummy_acc * 100,
    'Logistic\nRegression': lr_acc * 100,
    'ANN\n(Our Model)': test_acc * 100
}

fig, ax = plt.subplots(figsize=(8, 5))
bar_colors = ['#95a5a6', '#f39c12', '#2ecc71']
bars = ax.bar(results.keys(), results.values(),
              color=bar_colors, edgecolor='black', width=0.5, alpha=0.9)

for bar, val in zip(bars, results.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')

ax.set_ylim(50, 105)
ax.set_ylabel('Test Accuracy (%)', fontsize=11)
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.axhline(90, color='red', linestyle='--', lw=1.5, alpha=0.5, label='90% line')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nDummy classifier   : {dummy_acc*100:.1f}%")
print(f"Logistic Regression: {lr_acc*100:.1f}%")
print(f"Our ANN            : {test_acc*100:.1f}%")

---
## 🧪 Step 9: Make a Prediction on New Data

Let's use our trained model to predict a **single new patient sample** — just like it would be used in practice.

In [ ]:
# Pick a real test sample and pretend it's a new patient
sample_idx = 10  # change this to try different samples
new_patient = X_test[sample_idx].reshape(1, -1)
true_label = y_test[sample_idx]

# Scale and predict
new_patient_scaled = scaler.transform(new_patient)
prob = model.predict(new_patient_scaled, verbose=0)[0][0]
predicted = 'Benign' if prob >= 0.5 else 'Malignant'
actual = 'Benign' if true_label == 1 else 'Malignant'

# Visualize
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.axis('off')

bar_color = '#2ecc71' if predicted == 'Benign' else '#e74c3c'
ax.barh(['Malignant', 'Benign'], [1 - prob, prob],
        color=['#e74c3c', '#2ecc71'], edgecolor='black', alpha=0.8, height=0.4)
ax.set_xlim(0, 1)
ax.set_title(f'Patient #{sample_idx} | Predicted: {predicted} | Actual: {actual}',
             fontsize=12, fontweight='bold',
             color='green' if predicted == actual else 'red')
ax.axis('on')
ax.axvline(0.5, color='black', lw=2, linestyle='--', label='Decision boundary')
ax.text(prob, 0.6, f'{prob:.1%}', ha='center', fontsize=12, fontweight='bold', color='#2c3e50')
ax.set_xlabel('Probability', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Probability of Benign : {prob*100:.1f}%")
print(f"Prediction            : {predicted}")
print(f"True Label            : {actual}")
print(f"Correct?              : {'✅ Yes' if predicted == actual else '❌ No'}")

---
## 🎓 Step 10: Key Takeaways & Concept Summary

Let's cement what we learned with a final visual recap.

In [ ]:
# ─── Gradient Descent intuition ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Plot 1: Loss landscape (simplified 1D)
w = np.linspace(-3, 3, 300)
loss_curve = w**2 + 0.5 * np.sin(5 * w)  # artificial bowl-shaped loss

# Simulate gradient descent steps
w_curr = 2.5
lr_gd = 0.3
path_w, path_l = [w_curr], [w_curr**2 + 0.5 * np.sin(5 * w_curr)]
for _ in range(8):
    grad = 2 * w_curr + 2.5 * np.cos(5 * w_curr)
    w_curr -= lr_gd * grad
    path_w.append(w_curr)
    path_l.append(w_curr**2 + 0.5 * np.sin(5 * w_curr))

axes[0].plot(w, loss_curve, color='#3498db', lw=2.5, label='Loss Surface')
axes[0].plot(path_w, path_l, 'o--', color='#e74c3c', lw=1.5,
             markersize=8, label='Gradient Descent Steps')
for i, (pw, pl) in enumerate(zip(path_w, path_l)):
    axes[0].annotate(f'{i}', (pw, pl), textcoords='offset points',
                     xytext=(5, 5), fontsize=8, color='#c0392b')
axes[0].set_xlabel('Weight Value (w)')
axes[0].set_ylabel('Loss')
axes[0].set_title('Gradient Descent (Simplified)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Forward vs Backward pass
ax2 = axes[1]
ax2.axis('off')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 6)

boxes = [(1.5, 3, 'Input\nData', '#3498db'),
         (4.0, 3, 'ANN\nModel', '#e67e22'),
         (6.5, 3, 'Prediction', '#9b59b6'),
         (8.5, 3, 'Loss\n(Error)', '#e74c3c')]

for x, y, txt, col in boxes:
    rect = mpatches.FancyBboxPatch((x - 0.8, y - 0.7), 1.6, 1.4,
                                   boxstyle='round,pad=0.1', color=col,
                                   alpha=0.85, zorder=3)
    ax2.add_patch(rect)
    ax2.text(x, y, txt, ha='center', va='center', color='white',
             fontsize=9, fontweight='bold')

# Forward arrows
for (x1, _, __, ___), (x2, _, __, ____) in zip(boxes[:-1], boxes[1:]):
    ax2.annotate('', xy=(x2 - 0.8, 3.2), xytext=(x1 + 0.8, 3.2),
                 arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

# Backward arrows
ax2.annotate('', xy=(4.8, 2.7), xytext=(7.7, 2.7),
             arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2,
                             connectionstyle='arc3,rad=0.4'))
ax2.text(5.5, 1.5, '← Backpropagation (update weights)',
         ha='center', fontsize=9, color='#e74c3c', fontweight='bold')
ax2.text(4.5, 4.5, 'Forward Pass →', ha='center', fontsize=9,
         color='#27ae60', fontweight='bold')
ax2.set_title('Forward Pass & Backpropagation', fontsize=12, fontweight='bold')

plt.suptitle('How Neural Networks Learn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 📝 Summary

| Concept | What We Did |
|---|---|
| **Dataset** | Breast Cancer Wisconsin (569 samples, 30 features) |
| **Preprocessing** | StandardScaler (mean=0, std=1) |
| **Architecture** | 30 → 16 → 8 → 1 (641 trainable parameters) |
| **Activations** | ReLU (hidden), Sigmoid (output) |
| **Loss Function** | Binary Crossentropy |
| **Optimizer** | Adam |
| **Regularization** | Dropout (0.2) + Early Stopping |
| **Performance** | >95% accuracy, AUC > 0.99 |

---

## 🚀 What to Try Next (Exercises)

1. **Change the architecture**: Add more layers or neurons. Does accuracy improve or do you overfit?
2. **Change the activation**: Replace `relu` with `tanh`. How does training differ?
3. **Remove Dropout**: Retrain without Dropout. Do you see overfitting in the loss curves?
4. **Try a different threshold**: Instead of 0.5, use 0.3 for the Malignant decision. How does the confusion matrix change?
5. **Try the MNIST dataset**: Apply the same pipeline to handwritten digits (10-class classification).

---
*Built with TensorFlow/Keras, scikit-learn, and matplotlib.*